In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [21]:
filename = 'data/final_generation_minimized.csv'
df = pd.read_csv(filename)


In [22]:
start_date = "2025-01-01 01:00:00"
end_date   = "2026-04-19 02:00:00"

df["DateTime(UTC)"] = pd.to_datetime(df["DateTime(UTC)"])
df = df.sort_values("DateTime(UTC)")
df = df.set_index("DateTime(UTC)")
df_truncated = df.loc[start_date:end_date]
df_truncated = df_truncated.reset_index()


In [23]:
df_truncated['NetGeneration[MW]'] = df_truncated['ActualGenerationOutput[MW]'] - df_truncated['ActualConsumption[MW]']


In [24]:
# ---- Basic time features ----
df_truncated["day_of_week"] = df_truncated["DateTime(UTC)"].dt.dayofweek
df_truncated["day_of_year"] = df_truncated["DateTime(UTC)"].dt.dayofyear
df_truncated["month"] = df_truncated["DateTime(UTC)"].dt.month
df_truncated["year"] = df_truncated["DateTime(UTC)"].dt.year

# ---- Hour + quarter-hour ----
df_truncated["hour"] = df_truncated["DateTime(UTC)"].dt.hour

# quarter of hour: 0,1,2,3
df_truncated["quarter_hour"] = df_truncated["DateTime(UTC)"].dt.minute // 15

# ---- German public holidays ----
years = df_truncated["year"].unique()
de_holidays = holidays.Germany(years=years)

df_truncated["is_holiday"] = df_truncated["DateTime(UTC)"].dt.floor("D").isin(de_holidays)

# ---- Bridge day (Brückentag) ----
# A bridge day is typically:
# - Monday before a Tuesday holiday
# - Friday after a Thursday holiday

df_truncated["date"] = df_truncated["DateTime(UTC)"].dt.date

holidays_set = set(de_holidays)

def is_bridge_day(date):
    weekday = date.weekday()
    
    # Monday before Tuesday holiday
    if weekday == 0 and (date + pd.Timedelta(days=1)) in holidays_set:
        return 1
    
    # Friday after Thursday holiday
    if weekday == 4 and (date - pd.Timedelta(days=1)) in holidays_set:
        return 1
    
    return 0

df_truncated["is_bridge_day"] = df_truncated["date"].apply(is_bridge_day)

# convert holiday boolean to int
df_truncated["is_holiday"] = df_truncated["is_holiday"].astype(int)

/tmp/ipykernel_5519/1771040005.py:17: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  df_truncated["is_holiday"] = df_truncated["DateTime(UTC)"].dt.floor("D").isin(de_holidays)


In [25]:
# Rolling mean
df_truncated["netgen_roll_mean_4"] = df_truncated["NetGeneration[MW]"].rolling(4).mean()
df_truncated["netgen_roll_mean_24"] = df_truncated["NetGeneration[MW]"].rolling(24).mean()
df_truncated["netgen_roll_mean_96"] = df_truncated["NetGeneration[MW]"].rolling(96).mean()
df_truncated["netgen_roll_mean_672"] = df_truncated["NetGeneration[MW]"].rolling(672).mean()

# Rolling std
df_truncated["netgen_roll_std_4"] = df_truncated["NetGeneration[MW]"].rolling(4).std()
df_truncated["netgen_roll_std_24"] = df_truncated["NetGeneration[MW]"].rolling(24).std()
df_truncated["netgen_roll_std_96"] = df_truncated["NetGeneration[MW]"].rolling(96).std()

df_truncated["netgen_lag_1"] = df_truncated["NetGeneration[MW]"].shift(1)
df_truncated["netgen_lag_4"] = df_truncated["NetGeneration[MW]"].shift(4)
df_truncated["netgen_lag_12"] = df_truncated["NetGeneration[MW]"].shift(12)   # 3 hours
df_truncated["netgen_lag_24"] = df_truncated["NetGeneration[MW]"].shift(24)   # 6 hours
df_truncated["netgen_lag_96"] = df_truncated["NetGeneration[MW]"].shift(96)   # 1 day
df_truncated["netgen_lag_672"] = df_truncated["NetGeneration[MW]"].shift(672) # 1 week



In [26]:
# Net Generation data is done now. Now process target column
df_y = pd.read_csv('data/combined_energy_price_clean.csv', sep='\t')
df_y["DateTime(UTC)"] = pd.to_datetime(df_y["DateTime(UTC)"])
df_y = df_y.sort_values("DateTime(UTC)")
df_y = df_y.set_index("DateTime(UTC)")
df_y_truncated = df_y.loc[start_date:end_date]
df_y_truncated = df_y_truncated.reset_index()
df_y_truncated = df_y_truncated[df_y_truncated["Sequence"] != 1]

/tmp/ipykernel_5519/1455107288.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_y = pd.read_csv('data/combined_energy_price_clean.csv', sep='\t')


In [27]:
#both dataframes have the same datetime, we can combine them 
df_merged = df_truncated.merge(
    df_y_truncated[["DateTime(UTC)", "Price[Currency/MWh]"]],
    on="DateTime(UTC)",
    how="left"
)
df_merged.rename(columns={"NetGeneration[MW]": "NetGeneration_MW"}, inplace=True)
df_merged.head()

,DateTime(UTC),ActualGenerationOutput[MW],ActualConsumption[MW],NetGeneration_MW,day_of_week,day_of_year,month,year,hour,quarter_hour,...,netgen_roll_std_4,netgen_roll_std_24,netgen_roll_std_96,netgen_lag_1,netgen_lag_4,netgen_lag_12,netgen_lag_24,netgen_lag_96,netgen_lag_672,Price[Currency/MWh]
0,2025-01-01 01:00:00,53794.24,1625.80,52168.44,2,1,1,2025,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.60
1,2025-01-01 01:15:00,53889.47,2103.82,51785.65,2,1,1,2025,1,1,...,NaN,NaN,NaN,52168.44,NaN,NaN,NaN,NaN,NaN,-0.50
2,2025-01-01 01:30:00,53642.49,2521.47,51121.02,2,1,1,2025,1,2,...,NaN,NaN,NaN,51785.65,NaN,NaN,NaN,NaN,NaN,-6.51
3,2025-01-01 01:45:00,53483.38,2749.03,50734.35,2,1,1,2025,1,3,...,645.284781,NaN,NaN,51121.02,NaN,NaN,NaN,NaN,NaN,-3.50
4,2025-01-01 02:00:00,52040.88,2225.98,49814.90,2,1,1,2025,2,0,...,823.188503,NaN,NaN,50734.35,52168.44,NaN,NaN,NaN,NaN,-1.05


In [28]:
df_model = df_merged.sort_values("DateTime(UTC)").copy()
df_model["target_288"] = df_model["Price[Currency/MWh]"].shift(-288)
#df_model = df_model.dropna().reset_index(drop=True)

In [13]:
columns_to_drop = [
    "DateTime(UTC)",
    "ActualGenerationOutput[MW]",
    "ActualConsumption[MW]",
    "date"
]
df_model = df_model.drop(columns=columns_to_drop)
df_model.to_csv("feature_xgboost_ver1.csv", index=False)

In [29]:
columns_to_drop = [
    "ActualGenerationOutput[MW]",
    "ActualConsumption[MW]",
    "date"
]
df_model = df_model.drop(columns=columns_to_drop)
df_model.to_csv("feature_xgboost_with_UTC_ver1.csv", index=False)

In [17]:
df_model.describe().T

,count,mean,std,min,25%,50%,75%,max
NetGeneration_MW,45413.0,40581.868008,1.072100e+06,-1.783079e+08,42132.960000,50384.839506,58168.520000,8.226439e+04
day_of_week,45413.0,3.004910,1.995039e+00,0.000000e+00,1.000000,3.000000,5.000000,6.000000e+00
day_of_year,45413.0,153.664215,1.081466e+02,1.000000e+00,60.000000,129.000000,247.000000,3.650000e+02
month,45413.0,5.568802,3.539226e+00,1.000000e+00,3.000000,5.000000,9.000000,1.200000e+01
year,45413.0,2025.228503,4.198728e-01,2.025000e+03,2025.000000,2025.000000,2025.000000,2.026000e+03
hour,45413.0,11.498866,6.922727e+00,0.000000e+00,5.000000,11.000000,17.000000,2.300000e+01
quarter_hour,45413.0,1.499967,1.118056e+00,0.000000e+00,0.000000,1.000000,2.000000,3.000000e+00
is_holiday,45413.0,0.025279,1.569733e-01,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00
is_bridge_day,45413.0,0.008456,9.156644e-02,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00
netgen_roll_mean_4,45410.0,40581.939545,6.149687e+05,-7.257449e+07,42165.243273,50408.900271,58144.293434,8.185607e+04


In [19]:
df = df_model.copy()
filtered_df = df[df["Price[Currency/MWh]"] < -150]
filtered_df

,NetGeneration_MW,day_of_week,day_of_year,month,year,hour,quarter_hour,is_holiday,is_bridge_day,netgen_roll_mean_4,...,netgen_roll_std_24,netgen_roll_std_96,netgen_lag_1,netgen_lag_4,netgen_lag_12,netgen_lag_24,netgen_lag_96,netgen_lag_672,Price[Currency/MWh],target_288
7711,61169.58834,5,81,3,2025,8,3,0,0,59949.823967,...,4760.947758,5114.939854,60129.639312,58031.192408,50077.515628,47743.260668,54415.597964,49798.346280,-235.61,70.80
8663,56772.73000,1,91,4,2025,6,3,0,0,55928.485000,...,7142.601156,7766.983017,56569.170000,53382.000000,43322.840000,36867.860000,59901.347892,52585.777736,-200.00,-499.99
8667,59133.48000,1,91,4,2025,7,3,0,0,58859.477500,...,7609.953614,7632.180379,59030.320000,56772.730000,49528.790000,37824.190000,61201.196680,55032.348840,-200.00,10.01
8792,60025.37000,2,92,4,2025,15,0,0,0,61014.110000,...,1065.950362,7929.761325,60374.090000,61993.580000,63072.950000,63394.980000,53047.160000,53153.979852,-194.44,-100.00
8855,58241.09000,3,93,4,2025,6,3,0,0,58126.982500,...,6306.645618,7638.239866,58734.720000,56304.110000,48845.570000,41760.510000,62588.070000,53942.115588,-200.00,-29.97
8859,58310.69000,3,93,4,2025,7,3,0,0,58008.460000,...,5916.804611,7583.152034,57915.810000,58241.090000,52690.390000,41451.120000,59831.720000,56997.862412,-200.00,-54.99
8951,53624.19000,4,94,4,2025,6,3,0,0,52743.847500,...,6017.073783,8780.200472,53275.640000,49589.280000,40408.120000,37076.450000,58241.090000,59284.794048,-499.99,50.03
11564,48724.44000,3,121,5,2025,12,0,1,0,49733.567500,...,4989.699946,9394.865173,49531.580000,50999.270000,49643.780000,34754.160000,56834.960000,47823.160000,-160.55,-34.97
12522,51535.88000,6,131,5,2025,11,2,0,0,52237.887500,...,5987.769689,8501.634745,52045.400000,53260.260000,51658.390000,34993.230000,50213.640000,47103.690000,-169.64,-64.95
12523,50425.06000,6,131,5,2025,11,3,0,0,51623.245000,...,5345.049464,8513.818665,51535.880000,52883.630000,52861.660000,35917.930000,49567.520000,46860.770000,-169.90,-59.93
